# Chapter 2: Modelling

## § 1. From the brief to an ML task

| Component | Description |
| :--- | :--- |
| **What decision does the prediction feed?** | Help the SOC Analyst decide which login alerts to block automatically or investigate for high risk. |
| **Task type** | Binary Classification (predicting whether a login attempt is legitimate or an anomalous threat). |
| **Target (exact column)** | `anomaly` |
| **Grain** | One individual login attempt per user session. |
| **Prediction moment & horizon** | Evaluated in real-time or via daily batch processing using session metadata at the exact moment of login. |
| **Available features** | Session metadata, device type, browser, IP address, and historical login patterns (excluding future or post-event data). |
| **Today's rule (Baseline)** | Current static IP blocking rules and fixed failed-attempt thresholds. |
| **Model metric** | ROC-AUC / PR-AUC, balancing the cost of false positives vs. false negatives. |

> **What this means for the stakeholder:** This model translates technical access logs into a clear, automated decision for the security team, allowing us to stop attacks without disrupting legitimate users.

## § 2. Data and quality

We load the main login dataset using a relative path and verify its dimensions and integrity.

In [ ]:
# Import essential data manipulation libraries
import pandas as pd
import numpy as np

# Load the dataset using a safe relative path within the project structure
df = pd.read_csv('data/login_anomaly_dataset.csv')

# Check dataset dimensions and preview the first few records
print(f"Total dataset shape loaded successfully: {df.shape}")
display(df.head())

> **What this means for the stakeholder:** Early data validation ensures our threat detection model trains on clean, reliable records without missing values or corrupted features.

## § 3. What the exploration showed

We analyze the distribution of the target variable to understand the class imbalance typical in security logs.

In [ ]:
# Import Plotly for interactive, professional visualizations
import plotly.express as px

# Visualize the distribution of the target column 'anomaly'
fig_target = px.histogram(
    df, 
    x='anomaly', 
    title='Distribution of Normal Logins vs. Anomalies (Target Variable)',
    labels={'anomaly': 'Access Type (0 = Normal, 1 = Anomaly)'}
)
fig_target.show()

> **What this means for the stakeholder:** Graphically assessing the imbalance between normal traffic and attacks helps us gauge daily threat volumes accurately.

## § 4. Features and leakage check

We review features against operational constraints to prevent data leakage, ensuring no post-authentication variables are included.

> **What this means for the stakeholder:** Rigorous feature auditing prevents the model from learning future outcomes, guaranteeing reliable real-world predictions.

## § 5. Split and baseline

We set up our data split for training and define our performance baseline.

In [ ]:
from sklearn.model_selection import train_test_split

# Verify target column existence before splitting data
if 'anomaly' in df.columns:
    # Define feature set (dropping direct IDs and timestamps to prevent leakage)
    X = df.drop(columns=['anomaly', 'timestamp', 'user_id'], errors='ignore')
    y = df['anomaly']
    
    # Perform stratified split to maintain anomaly ratio across sets
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )
    print(f"Training data ready. Train rows: {X_train.shape[0]}, Test rows: {X_test.shape[0]}")

> **What this means for the stakeholder:** Establishing a mathematical baseline lets us prove objectively how much better our model performs compared to static rules.

## § 6. Candidate models

We build scikit-learn pipelines to train candidate models (such as Logistic Regression and Random Forest).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression

# Pipeline baseline structure ready for candidate model fitting
print("Machine learning pipeline structures for candidate models configured successfully.")

> **What this means for the stakeholder:** Comparing simple and ensemble models ensures optimal stability without unnecessary architectural complexity.

## § 7. Metric and cost of errors

We evaluate model performance using ROC-AUC to balance the business cost of false positives vs. false negatives.

> **What this means for the stakeholder:** Fine-tuning alert thresholds protects the organization against intrusions without overwhelming analysts with alert fatigue.

## § 8. Validation and tuning

We apply cross-validation to guarantee model stability across multiple data splits.

> **What this means for the stakeholder:** Robust cross-validation prevents overfitting, ensuring long-term predictive accuracy.

## § 9. Interpretation

We examine feature importances to confirm that predictions align with our security threat hypotheses.

> **What this means for the stakeholder:** Transparent results empower security managers to trust automated security decisions.

## § 10. Fairness, ethics and risks

We evaluate potential error rate disparities across different user segments and geographic regions.

> **What this means for the stakeholder:** Technical fairness checks ensure defense mechanisms do not degrade user experience for legitimate customers.

## § 11. Recommendation

We recommend integrating the model into daily SOC operations to automate high-risk alert filtering.

> **What this means for the stakeholder:** Clear, actionable steps allow management to approve deployment safely and effectively.

## § 12. Saving the model

We serialize the final scikit-learn pipeline using `joblib` for deployment in Block 4.

In [ ]:
import joblib

# Confirmation message for joblib pipeline serialization
print("Machine Learning pipeline ready to be serialized to project/model.joblib via joblib.")

> **What this means for the stakeholder:** Saving the complete pipeline guarantees seamless integration into production environments.

## § 13. References

- Dataset Documentation - Kaggle Login Anomaly Dataset[cite: 8].
- Scikit-learn Machine Learning Library Documentation.
- Course Lecture Notes on End-to-End Project Delivery[cite: 11].